In [1]:
# We want to compare one dot using only canonical povm, sicpovm

# and two dots using choi, no choi, sicpovm or separable

In [2]:
import numpy as np
import jax
import jax.numpy as jnp

jax.config.update("jax_platform_name", "cpu")
from jax import vmap, jit
import matplotlib.pyplot as plt
import joblib
from qdots_qll.models import game
from qdots_qll import all_funcs
import seaborn as sns
import pandas as pd
import scipy
from functools import reduce
import os
import re
from scipy.stats import binned_statistic

import seaborn as sns
from qbism import sic_povm


import matplotlib.font_manager as font_manager

import equinox as eqx

# from matplotlib import rcParams
from scipy.stats import binned_statistic


font = {"family": "Inter"}  # , 'weight': 'normal', 'size': 12}

# Aplica la fuente definida a Matplotlib
plt.rc("font", **font)

sns.set_palette("colorblind")

In [3]:
names_four = [
    "$\\gamma ( - \\eta)$",
    "$\\gamma ( + \\eta)$",
    "$S ( - \\eta)$",
    "$S ( +\\eta)$",
]

names_three = [
    "$\\gamma ( + \\eta)$",
    "$S ( - \\eta)$",
    "$S ( +\\eta)$",
]

# One dot three params

In [4]:
det_times_list = []
fim_times_list = []

In [5]:
import qutip as qt
from qdots_qll.utils.povms import sigmas_povm

from qdots_qll.models.models_scratch_for_drafting import SingleQDot3Params


model = SingleQDot3Params(POVM_array=jnp.array(sigmas_povm))
ground_state_qdot = jnp.array(qt.ket2dm(qt.basis(2, 0))).flatten()

true_pars_singleqdot = jnp.array(
    [0.35833, 0.053851, -0.333695],
)

m = model

times = jnp.linspace(0, 60, 500)

fim_times = jax.vmap(
    lambda t: m.fim(true_pars_singleqdot, t, ground_state_qdot)
)(times)

fim_times_list.append(fim_times)
det_times = jax.vmap(lambda a: jnp.linalg.det(a))(fim_times[:, 0:, 0:])

det_times_list.append(det_times)

In [6]:
# fig, axs = plt.subplots(2, 1, dpi=200)

# for i in range(3):
#     axs.flatten()[0].plot(times, fim_times[:, i, i], label=names_three[i])


# ax = axs.flatten()[0]
# ax.legend()
# ax.set_title("Components of FIM")
# ax.set_xlabel("t (ps)")

# # plt.legend()

# ax = axs.flatten()[1]
# ax.plot(times, det_times)
# ax.set_title("$| F_{ij} |$")
# ax.set_xlabel("t (ps)")


# plt.tight_layout()
# plt.show()

In [7]:
import qutip as qt
from qdots_qll.utils.povms import sigmas_povm

from qdots_qll.models.models_scratch_for_drafting import SingleQDot3Params


model = SingleQDot3Params(POVM_array=jnp.array(sic_povm(2)))
ground_state_qdot = jnp.array(qt.ket2dm(qt.basis(2, 0))).flatten()

true_pars_singleqdot = jnp.array(
    [0.35833, 0.053851, -0.333695],
)

m = model

times = jnp.linspace(0, 60, 500)

fim_times = jax.vmap(
    lambda t: m.fim(true_pars_singleqdot, t, ground_state_qdot)
)(times)

fim_times_list.append(fim_times)

det_times = jax.vmap(lambda a: jnp.linalg.det(a))(fim_times[:, 0:, 0:])
det_times_list.append(det_times)


# fig, axs = plt.subplots(2, 1, dpi=200)

# for i in range(3):
#     axs.flatten()[0].plot(times, fim_times[:, i, i], label=names_three[i])


# ax = axs.flatten()[0]
# ax.legend()
# ax.set_title("Components of FIM")
# ax.set_xlabel("t (ps)")

# # plt.legend()

# ax = axs.flatten()[1]
# ax.plot(times, det_times)
# ax.set_title("$| F_{ij} |$")
# ax.set_xlabel("t (ps)")


# plt.tight_layout()
# plt.show()

In [8]:
# fig, axs = plt.subplots(3, 1, dpi=200)

# axs = axs.flatten()

# povm_list = ["canonical", "sicpovm"]
# for i, ax in enumerate(axs):
#     for j in range(len(fim_times_list)):
#         fim_times = fim_times_list[j]
#         ax.plot(times, fim_times[:, i, i], label=povm_list[j])
#         ax.set_xlabel("t (ps)")
#         ax.set_title(names_three[i])
# ax.legend()

# plt.tight_layout()

# plt.show()

In [9]:
# fig, ax = plt.subplots(1, 1, dpi=200)


# povm_list = ["canonical", "sicpovm"]

# for j in range(len(det_times_list)):
#     det_times = det_times_list[j]
#     ax.plot(times, det_times, label=povm_list[j])
#     ax.set_xlabel("t (ps)")
#     # ax.set_title(names_three[i])
# ax.legend()

# plt.tight_layout()

# plt.show()

# Two dots

# No choi

In [10]:
# det_times_list = []
# fim_times_list = []

In [12]:
from functools import reduce

from qdots_qll.models.game import vec
from itertools import product


list_ket_ii = [qt.tensor(qt.basis(2, i), qt.basis(2, i)) for i in range(2)]

ket_ii = reduce(lambda i, j: i + j, list_ket_ii).unit()

rho_omega_super = vec(qt.ket2dm(ket_ii).full())

povm1 = (
    jnp.array(
        [
            [0.5 * (qt.identity(2) + mat), 0.5 * (qt.identity(2) - mat)]
            for mat in [qt.sigmax(), qt.sigmay(), qt.sigmaz()]
        ]
    ).reshape(-1, 2, 2)
    / 3
)

joint_povm_local_measurements_povm1 = jnp.array(
    [jnp.kron(i[0], i[1]) for i in product(povm1, povm1)]
)

In [13]:
from qdots_qll.models.models_scratch_for_drafting import (
    two_qdots_separable_maps,
)

true_pars_twodots = game.true_pars

m = two_qdots_separable_maps(
    POVM_array=jnp.array(joint_povm_local_measurements_povm1)
)


times = jnp.linspace(0, 60, 500)

fim_times = jax.vmap(lambda t: m.fim(true_pars_twodots, t, rho_omega_super))(
    times
)


fim_times_list.append(fim_times)

det_times = jax.vmap(lambda a: jnp.linalg.det(a))(fim_times[:, 0:, 0:])
det_times_list.append(det_times)


true_pars_twodots = game.true_pars

m = two_qdots_separable_maps(POVM_array=jnp.array(sic_povm(4)))


times = jnp.linspace(0, 60, 500)

fim_times = jax.vmap(lambda t: m.fim(true_pars_twodots, t, rho_omega_super))(
    times
)


fim_times_list.append(fim_times)

det_times = jax.vmap(lambda a: jnp.linalg.det(a))(fim_times[:, 0:, 0:])
det_times_list.append(det_times)

In [14]:
fig, axs = plt.subplots(2, 2, dpi=200)

axs = axs.flatten()

povm_list = ["canonical", "sicpovm"]
for i, ax in enumerate(axs):
    for j in range(len(fim_times_list)):
        fim_times = fim_times_list[j]
        ax.plot(times, fim_times[:, i, i], label=povm_list[j])
        ax.set_xlabel("t (ps)")
        ax.set_title(names_four[i])
ax.legend()

plt.tight_layout()

plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1, dpi=200)

# axs = axs.flatten()

povm_list = ["canonical", "sicpovm"]

for j in range(len(det_times_list)):
    det_times = det_times_list[j]
    ax.plot(times, det_times, label=povm_list[j])
    ax.set_xlabel("t (ps)")
    # ax.set_title(names_three[i])
ax.legend()

plt.tight_layout()

plt.show()

# Choi case

In [15]:
det_times_list = []
fim_times_list = []

In [16]:
from qdots_qll.models.models_scratch_for_drafting import (
    two_qdots_identity_for_systemB,
)

from qdots_qll.models.game import vec


true_pars_twodots = game.true_pars

m = two_qdots_identity_for_systemB(
    POVM_array=jnp.array(joint_povm_local_measurements_povm1)
)

times = jnp.linspace(0, 60, 500)

fim_times = jax.vmap(lambda t: m.fim(true_pars_twodots, t, rho_omega_super))(
    times
)


fim_times_list.append(fim_times)

det_times = jax.vmap(lambda a: jnp.linalg.det(a))(fim_times[:, 0:, 0:])
det_times_list.append(det_times)


true_pars_twodots = game.true_pars

m = two_qdots_identity_for_systemB(POVM_array=jnp.array(sic_povm(4)))


times = jnp.linspace(0, 60, 500)

fim_times = jax.vmap(lambda t: m.fim(true_pars_twodots, t, rho_omega_super))(
    times
)


fim_times_list.append(fim_times)

det_times = jax.vmap(lambda a: jnp.linalg.det(a))(fim_times[:, 0:, 0:])
det_times_list.append(det_times)

In [27]:
fig, axs = plt.subplots(2, 2, dpi=200)

axs = axs.flatten()

povm_list = ["canonical", "sicpovm"]
for i, ax in enumerate(axs):
    for j in range(len(fim_times_list)):
        fim_times = fim_times_list[j]
        ax.plot(times, fim_times[:, i, i], label=povm_list[j])
        ax.set_xlabel("t (ps)")
        ax.set_title(names_four[i])
ax.legend()

plt.tight_layout()

plt.show()

In [28]:
fig, ax = plt.subplots(1, 1, dpi=200)

# axs = axs.flatten()

povm_list = ["canonical", "sicpovm"]

for j in range(len(det_times_list)):
    det_times = det_times_list[j]
    ax.plot(times, det_times, label=povm_list[j])
    ax.set_xlabel("t (ps)")
    # ax.set_title(names_three[i])
ax.legend()

plt.tight_layout()

plt.show()

In [8]:
# ALL of them

In [17]:
det_times_list = []
fim_times_list = []


import qutip as qt
from qdots_qll.utils.povms import sigmas_povm

from qdots_qll.models.models_scratch_for_drafting import SingleQDot3Params


model = SingleQDot3Params(POVM_array=jnp.array(sigmas_povm))
ground_state_qdot = jnp.array(qt.ket2dm(qt.basis(2, 0))).flatten()

true_pars_singleqdot = jnp.array(
    [0.35833, 0.053851, -0.333695],
)

m = model

times = jnp.linspace(0, 60, 500)

fim_times = jax.vmap(
    lambda t: m.fim(true_pars_singleqdot, t, ground_state_qdot)
)(times)

fim_times_list.append(fim_times)
det_times = jax.vmap(lambda a: jnp.linalg.det(a))(fim_times[:, 0:, 0:])

det_times_list.append(det_times)


import qutip as qt
from qdots_qll.utils.povms import sigmas_povm

from qdots_qll.models.models_scratch_for_drafting import SingleQDot3Params


model = SingleQDot3Params(POVM_array=jnp.array(sic_povm(2)))
ground_state_qdot = jnp.array(qt.ket2dm(qt.basis(2, 0))).flatten()

true_pars_singleqdot = jnp.array(
    [0.35833, 0.053851, -0.333695],
)

m = model

times = jnp.linspace(0, 60, 500)

fim_times = jax.vmap(
    lambda t: m.fim(true_pars_singleqdot, t, ground_state_qdot)
)(times)

fim_times_list.append(fim_times)

det_times = jax.vmap(lambda a: jnp.linalg.det(a))(fim_times[:, 0:, 0:])
det_times_list.append(det_times)


from functools import reduce

from qdots_qll.models.game import vec
from itertools import product


list_ket_ii = [qt.tensor(qt.basis(2, i), qt.basis(2, i)) for i in range(2)]

ket_ii = reduce(lambda i, j: i + j, list_ket_ii).unit()

rho_omega_super = vec(qt.ket2dm(ket_ii).full())

povm1 = (
    jnp.array(
        [
            [0.5 * (qt.identity(2) + mat), 0.5 * (qt.identity(2) - mat)]
            for mat in [qt.sigmax(), qt.sigmay(), qt.sigmaz()]
        ]
    ).reshape(-1, 2, 2)
    / 3
)

joint_povm_local_measurements_povm1 = jnp.array(
    [jnp.kron(i[0], i[1]) for i in product(povm1, povm1)]
)


from qdots_qll.models.models_scratch_for_drafting import (
    two_qdots_separable_maps,
)

true_pars_twodots = game.true_pars

m = two_qdots_separable_maps(
    POVM_array=jnp.array(joint_povm_local_measurements_povm1)
)


times = jnp.linspace(0, 60, 500)

fim_times = jax.vmap(lambda t: m.fim(true_pars_twodots, t, rho_omega_super))(
    times
)


fim_times_list.append(fim_times)

det_times = jax.vmap(lambda a: jnp.linalg.det(a))(fim_times[:, 0:, 0:])
det_times_list.append(det_times)


true_pars_twodots = game.true_pars

m = two_qdots_separable_maps(POVM_array=jnp.array(sic_povm(4)))


times = jnp.linspace(0, 60, 500)

fim_times = jax.vmap(lambda t: m.fim(true_pars_twodots, t, rho_omega_super))(
    times
)


fim_times_list.append(fim_times)

det_times = jax.vmap(lambda a: jnp.linalg.det(a))(fim_times[:, 0:, 0:])
det_times_list.append(det_times)


from qdots_qll.models.models_scratch_for_drafting import (
    two_qdots_identity_for_systemB,
)

from qdots_qll.models.game import vec


true_pars_twodots = game.true_pars

m = two_qdots_identity_for_systemB(
    POVM_array=jnp.array(joint_povm_local_measurements_povm1)
)

times = jnp.linspace(0, 60, 500)

fim_times = jax.vmap(lambda t: m.fim(true_pars_twodots, t, rho_omega_super))(
    times
)


fim_times_list.append(fim_times)

det_times = jax.vmap(lambda a: jnp.linalg.det(a))(fim_times[:, 0:, 0:])
det_times_list.append(det_times)


true_pars_twodots = game.true_pars

m = two_qdots_identity_for_systemB(POVM_array=jnp.array(sic_povm(4)))


times = jnp.linspace(0, 60, 500)

fim_times = jax.vmap(lambda t: m.fim(true_pars_twodots, t, rho_omega_super))(
    times
)


fim_times_list.append(fim_times)

det_times = jax.vmap(lambda a: jnp.linalg.det(a))(fim_times[:, 0:, 0:])
det_times_list.append(det_times)

In [42]:
array_legends = [
    "POVM sigma-basis",
    "SIC-POVM",
    "No-Choi  local POVM sigma-basis",
    "No-Choi  SIC-POVM",
    "Choi  local POVM sigma-basis",
    "Choi  SIC-POVM",
]

In [74]:
linestyle_list = ["-." , "-", "-.", "-", "-.", "-"]

colors_list = ["salmon", "navy", "salmon", "salmon", "navy", "navy"]

In [75]:
fig, axs = plt.subplots(2, 2, figsize=(10, 10), dpi=600)

axs = axs.flatten()

# povm_list = ["canonical", "sicpovm"]
for i, ax in enumerate(axs):
    for j in range(2, 6, 1):
        fim_times = fim_times_list[j]
        ax.plot(times, fim_times[:, i, i], label=array_legends[j], linestyle=linestyle_list[j], color=colors_list[j])
        ax.set_xlabel("t (ps)")
        ax.set_title(names_four[i])
ax.legend()


plt.suptitle(
    "$F( \\theta_{True}, t)_{ii}$.\n Two dots"
)
plt.tight_layout()

plt.show()

In [76]:
fig, axs = plt.subplots(3, 1, figsize=(10, 10), dpi=600)

axs = axs.flatten()

# povm_list = ["canonical", "sicpovm"]
for i, ax in enumerate(axs):
    for j in range(0, 2, 1):
        fim_times = fim_times_list[j]
        ax.plot(times, fim_times[:, i, i], label=array_legends[j], linestyle=linestyle_list[j], color=colors_list[j])
        ax.set_xlabel("t (ps)")
        ax.set_title(names_three[i])
ax.legend()


plt.suptitle(
    "$F( \\theta_{True}, t)_{ii}$.\n Single qdot. Comparing POVM in sigma basis vs SIC-POVM"
)

plt.tight_layout()


plt.show()

In [83]:
fig, ax = plt.subplots(1, 1, dpi=200)

# axs = axs.flatten()

# povm_list = ["canonical", "sicpovm"]


# povm_list = ["canonical", "sicpovm"]
for j in range(0, 2, 1):
    det_times = det_times_list[j]
    ax.plot(
        times,
        det_times,
        label=array_legends[j],
        linestyle=linestyle_list[j],
        color=colors_list[j],
    )
    # ax.set_title(names_three[i])

    ax.set_xlabel("t (ps)")
ax.legend()


# plt.suptitle(
#     "$F( \\theta_{True}, t)_{ii}$.\n Single qdot. Comparing POVM in sigma basis vs SIC-POVM"
# )

plt.suptitle(
    "$\\operatorname{det} ( F( \\theta_{True}, t)_{ij} ) $.\n Single qdot. Comparing POVM in sigma basis vs SIC-POVM"
)

# ax.loglog()
plt.tight_layout()

plt.show()

In [82]:
fig, ax = plt.subplots(1, 1, dpi=200)

# axs = axs.flatten()

# povm_list = ["canonical", "sicpovm"]




# povm_list = ["canonical", "sicpovm"]
for j in range(2, 6, 1):
    det_times = det_times_list[j]
    ax.plot(times, det_times, label=array_legends[j], linestyle=linestyle_list[j], color=colors_list[j])
    ax.set_xlabel("t (ps)")
ax.legend()


plt.suptitle(
    "$\\operatorname{det} ( F( \\theta_{True}, t)_{ij} ) $.\n Two dots."
)
# ax.loglog()
plt.tight_layout()

plt.show()